# ¿A qué famoso me parezco?

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

class FaceEmbeddingModel(nn.Module):
    def __init__(
        self,
        embedding_dim=128
    ):
        super().__init__()

        self.backbone = models.resnet18(
            weights=
            models.ResNet18_Weights.DEFAULT
        )

        num_features = (self.backbone.fc.in_features)

        self.backbone.fc = nn.Sequential(
            nn.Linear(
                num_features,
                embedding_dim
            )
        )

    def forward(self, x):
        embedding = self.backbone(x)

        # Normalización L2
        embedding = F.normalize(
            embedding,
            p=2,
            dim=1
        )

        return embedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceEmbeddingModel(128)

checkpoint = torch.load("best_face_embedding_resnet18.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Carga del modelo
dnn_model = "deploy.prototxt.txt"
dnn_weights = "res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(dnn_model, dnn_weights)

gallery = torch.load("gallery_embeddings.pt", map_location="cpu")

def get_most_similar_famous(face):
    gallery_embeddings = gallery["embeddings"].to(device)
    filenames = gallery["filenames"]

    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face_image = Image.fromarray(face_rgb)

    face_tensor = transform(face_image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        query_embedding = model(face_tensor)
        query_embedding = F.normalize(query_embedding, p=2, dim=1)

    similarities = query_embedding @ gallery_embeddings.T

    best_index = similarities.argmax(dim=1).item()

    return "."+filenames[best_index]


video_capture = cv2.VideoCapture(0)

while True:
    ret, frame = video_capture.read()

    if not ret:
        break

    # Aplicamos el efecto espejo sobre la entrada
    frame=cv2.flip(frame, 1)

    H, W = frame.shape[:2]

    # DNN
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104.0, 117.0, 123.0], False, False)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > 0.5:
            box = detections[0, 0, i, 3:7] * np.array([W, H, W, H])

            (x1, y1, x2, y2) = box.astype("int")
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)

            face_crop = frame[y1:y2, x1:x2]

            if face_crop.size == 0:
                continue

            found = get_most_similar_famous(face_crop)
            if found is not None:
                # Imagen que queremos superponer
                overlay = cv2.imread(found)
                if overlay is None:
                    continue

                # Posición: centro zona superior
                h, w = overlay.shape[:2]

                x = max(0, min((W - w) // 2, W - w))
                y = max(0, min(20, H - h))
                frame[y:y+h, x:x+w] = overlay

    # Mostramos resultados
    cv2.putText(frame, "Presiona 'ESC' para salir",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

    cv2.imshow('Face Detection', frame)

    # 'Esc para salir"
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

# Cerramos
video_capture.release()
cv2.destroyAllWindows()